# Robust AIGC Image Detection — end-to-end on a free Colab GPU

**Run this top to bottom.** Every step writes its output to Google Drive, so a
disconnect costs you one cell, not the whole run.

| Stage | What it does | Time on a free T4 |
|---|---|---|
| 0 | GPU check, install, clone | 3 min |
| 1 | Download data | 10–25 min |
| 2 | Build manifest + shortcut audit | 3 min |
| 3 | Extract features (the only expensive step) | 15–35 min |
| 4 | Train both heads | under 1 min |
| 5 | Robustness grid | 15–25 min |
| 6 | Error analysis + figures | 3 min |
| 7 | Score the official demo set | 10 min |
| 8 | Launch the demo for the video | 2 min |

> **First time through, set `QUICK = True` in Stage 0.** It runs the identical
> pipeline on ~2,000 images in about 15 minutes total. Get a green run end to
> end, *then* set `QUICK = False` and do the real one. Never debug at full scale.

**Runtime → Change runtime type → T4 GPU** before you start.

## Stage 0 — environment

In [ ]:
#@title Check the GPU. If this says "no GPU", fix it before anything else.
import subprocess, sys
print(sys.version.split()[0])
try:
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"]).decode())
except Exception:
    print("!! NO GPU. Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then rerun.")

In [ ]:
#@title Settings — the only cell you normally edit
GITHUB_REPO = "https://github.com/YOUR_USERNAME/aigc-robust-detector.git"  #@param {type:"string"}

QUICK       = True    # True = ~2k images, ~15 min. Set False for the real run.
BACKBONE    = "clip-vit-l14"   # 304M-param vision tower, well under the 2B cap
N_VIEWS     = 4       # damaged copies per training image
USE_DRIVE   = True    # persist everything to Drive so a disconnect is survivable

# Full-run sizes. Chosen to fit comfortably in one Colab session with headroom.
TRAIN_PER_CLASS = 600 if QUICK else 7500
TEST_LIMIT      = 300 if QUICK else 2500

print(f"QUICK={QUICK} | train/class={TRAIN_PER_CLASS} | test={TEST_LIMIT} | views={N_VIEWS}")

In [ ]:
#@title Mount Drive and pick the working directory
import os, pathlib
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = pathlib.Path('/content/drive/MyDrive/aigc_hackathon')
else:
    WORK = pathlib.Path('/content/aigc_hackathon')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

for sub in ("data", "features", "checkpoints", "results", "docs"):
    (WORK / sub).mkdir(exist_ok=True)
print("working in", WORK)

In [ ]:
#@title Clone (or update) the repo and install
import subprocess, pathlib, os
REPO_DIR = pathlib.Path('/content/repo')
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, str(REPO_DIR)], check=True)

!pip -q install open_clip_torch tabulate gradio kagglehub huggingface_hub 2>&1 | tail -2

import sys
sys.path.insert(0, str(REPO_DIR / "src"))
os.environ["PYTHONPATH"] = str(REPO_DIR / "src")
import aigcdet
print("aigcdet", aigcdet.__version__, "from", REPO_DIR)

## Stage 1 — data

Three sources, three different jobs:

* **CIFAKE** (Kaggle) — tiny 32×32 images. Fast, and it is the *control*: if a
  detector cannot beat chance here something is broken in the pipeline. Do not
  report CIFAKE numbers as your headline result; the images are too small and
  too easy to say anything about real-world robustness.
* **SID_Set** (HuggingFace) — full-resolution real and fully-synthetic images.
  **This is your training and test set.**
* **COCO val2017 + DALL·E Advanced** — the organisers' demo set. Never trained
  on, scored once at the end, reported honestly.

Run 1a **or** 1b. 1a is the guaranteed-to-work smoke path; 1b is the real one.

In [ ]:
#@title 1a — CIFAKE (fast smoke path, ~5 min)
import kagglehub, pathlib
CIFAKE = pathlib.Path(kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images"))
print(CIFAKE)
!find "$CIFAKE" -maxdepth 3 -type d | head -20

In [ ]:
#@title 1b — SID_Set (the real training data)
# Downloads a subset. If the repo layout differs from what the adapter below
# expects, print the tree and adjust REAL_DIRS / FAKE_DIRS - do not fight the
# downloader.
from huggingface_hub import snapshot_download
import pathlib

SID = pathlib.Path(snapshot_download(
    repo_id="saberzl/SID_Set",
    repo_type="dataset",
    local_dir="/content/data/SID_Set",
    max_workers=8,
    # Start with a slice; drop allow_patterns for the whole thing (large).
    allow_patterns=["*.parquet", "*.json", "*/train-0000*", "*/train-0001*"],
))
print(SID)
!find "$SID" -maxdepth 3 | head -40

In [ ]:
#@title 1b-ii — If SID_Set arrived as parquet/arrow, unpack it to image folders
# HuggingFace image datasets often ship as parquet with embedded bytes. This
# writes them out as JPEGs in the real/ | fake/ layout the manifest builder
# expects, keeping the generator name as the folder so the family-disjoint
# split still works.
import pathlib, io
from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm

OUT = pathlib.Path('/content/data/sid_images'); OUT.mkdir(parents=True, exist_ok=True)
ds = load_dataset("saberzl/SID_Set", split="train", streaming=True)

# Inspect one record so you know what the columns are actually called.
first = next(iter(ds)); print({k: type(v).__name__ for k, v in first.items()})

LABEL_KEY = "label"          # <- adjust after looking at the printout above
IMAGE_KEY = "image"
# SID_Set labels: 0 real, 1 tampered, 2 fully synthetic. We keep 0 and 2 only:
# partially-tampered images are a different task (localisation), and mixing them
# in makes the binary label incoherent.
KEEP = {0: "real/photo", 2: "fake/sid_synthetic"}

per_class, cap = {}, TRAIN_PER_CLASS + TEST_LIMIT
for i, rec in enumerate(tqdm(ds, total=cap * 2)):
    lab = int(rec[LABEL_KEY])
    if lab not in KEEP:
        continue
    rel = KEEP[lab]
    n = per_class.get(rel, 0)
    if n >= cap:
        if all(per_class.get(v, 0) >= cap for v in KEEP.values()):
            break
        continue
    d = OUT / rel; d.mkdir(parents=True, exist_ok=True)
    img = rec[IMAGE_KEY]
    if isinstance(img, dict):
        img = Image.open(io.BytesIO(img["bytes"]))
    img.convert("RGB").save(d / f"{n:06d}.jpg", quality=95)
    per_class[rel] = n + 1

print(per_class)

In [ ]:
#@title 1c — the organisers' demo set (COCO val2017 + DALL·E Advanced). NEVER trained on.
import pathlib
DEMO = pathlib.Path('/content/data/demo'); DEMO.mkdir(parents=True, exist_ok=True)

# COCO val2017 = the 4,998 authentic images
!wget -q -c http://images.cocodataset.org/zips/val2017.zip -O /content/val2017.zip
!unzip -q -o /content/val2017.zip -d "$DEMO/real" && ls "$DEMO/real/val2017" | head -3

# DALL-E Advanced subset of WildFake: download from ModelScope (use the page's
# translate button) and unzip into $DEMO/fake/dalle_advanced/.
(DEMO / "fake" / "dalle_advanced").mkdir(parents=True, exist_ok=True)
print(f"Put the DALL-E Advanced images in: {DEMO/'fake'/'dalle_advanced'}")
!find "$DEMO" -maxdepth 3 -type d

## Stage 2 — manifest and the shortcut audit

The audit is the cell that separates a credible submission from an
accidentally-cheating one. It trains a classifier on **file metadata only** —
dimensions, file size, format, JPEG quantisation tables. If that alone scores
well above chance, then some of any accuracy you report is not about the
picture. Read the verdict it prints and quote it in your README either way.

In [ ]:
#@title 2 — build the manifest with a generator-family-disjoint split
DATA_ROOT = "/content/data/sid_images"   # or str(CIFAKE / "train") for the CIFAKE path

!cd /content/repo && python scripts/build_manifest.py \
    --root "$DATA_ROOT" \
    --out {WORK}/data/manifest.csv \
    --limit_per_class {TRAIN_PER_CLASS + TEST_LIMIT} \
    --test_frac 0.25 \
    --source sid_set

import pandas as pd
man = pd.read_csv(WORK / "data/manifest.csv")
display(man.groupby(["split", "label", "family"]).size().to_frame("n"))

## Stage 3 — feature extraction

The only expensive step. Two banks from the same manifest:

* `train_clean.npz` — one pristine embedding per image. **The control.**
* `train_aug.npz` — `N_VIEWS` independently damaged copies per image. **The treatment.**

Everything after this is seconds, which is why you can afford to experiment.

In [ ]:
#@title 3a — clean features (the control)
!cd /content/repo && python scripts/extract_features.py \
    --manifest {WORK}/data/manifest.csv --split train \
    --out {WORK}/features/train_clean.npz \
    --backbone {BACKBONE} --batch_size 64

In [ ]:
#@title 3b — augmented features (the treatment)
!cd /content/repo && python scripts/extract_features.py \
    --manifest {WORK}/data/manifest.csv --split train \
    --out {WORK}/features/train_aug.npz \
    --backbone {BACKBONE} --batch_size 64 --augment --n_views {N_VIEWS}

## Stage 4 — train both heads (seconds)

In [ ]:
#@title 4 — two models, one variable changed between them
!cd /content/repo && python scripts/train_head.py \
    --features {WORK}/features/train_clean.npz --out {WORK}/checkpoints/detector_clean.pt

!cd /content/repo && python scripts/train_head.py \
    --features {WORK}/features/train_aug.npz --out {WORK}/checkpoints/detector_robust.pt

import json
for name in ("clean", "robust"):
    m = json.load(open(WORK / f"checkpoints/detector_{name}.metrics.json"))
    print(f"{name:8s} val AUC {m['val_auc']:.4f} | ECE {m['val_ece']:.4f} | "
          f"threshold {m['threshold']['threshold']:.3f} | params {m['n_trainable_params']:,}")

## Stage 5 — the robustness grid (deliverable #4)

This is your headline result. It scores both models on the same test images
under every corruption, at each model's own fixed threshold. Nothing is re-tuned
per condition.

In [ ]:
#@title 5 — run the grid and generate the figures
!cd /content/repo && python scripts/evaluate_robustness.py \
    --manifest {WORK}/data/manifest.csv --split test \
    --checkpoints clean={WORK}/checkpoints/detector_clean.pt \
                  robust={WORK}/checkpoints/detector_robust.pt \
    --out {WORK}/results/ --limit {TEST_LIMIT} --batch_size 64

from IPython.display import Image as IPImage, display
display(IPImage(str(WORK / "results/robustness_auc.png")))
display(IPImage(str(WORK / "results/degradation_auc.png")))
print(open(WORK / "results/headline.txt").read())

## Stage 6 — error analysis (deliverable #5)

Look at the two contact sheets **as a team**, out loud, for half an hour. Write
what you actually see into the "What we would change" section of
`docs/ERROR_ANALYSIS.md`. That paragraph is worth more to the judges than
another point of AUC.

In [ ]:
#@title 6 — most confident errors
!cd /content/repo && python scripts/error_analysis.py \
    --scores {WORK}/results/scores.csv \
    --manifest {WORK}/data/manifest.csv \
    --model robust --out {WORK}/docs/ERROR_ANALYSIS.md \
    --figures_dir {WORK}/results

from IPython.display import Image as IPImage, display, Markdown
for f in ("false_positives.png", "false_negatives.png"):
    p = WORK / "results" / f
    if p.exists():
        display(IPImage(str(p)))
display(Markdown(open(WORK / "docs/ERROR_ANALYSIS.md").read()[:4000]))

## Stage 7 — the organisers' demo set

Scored once, at the end, with the model frozen. Whatever this says, report it.
A lower number here than on your own test split is the *expected* result and
saying so is a strength, not an admission — it is the generalisation gap the
whole write-up is about.

In [ ]:
#@title 7 — score COCO val2017 + DALL-E Advanced
!cd /content/repo && python scripts/build_manifest.py \
    --real_dir /content/data/demo/real/val2017 \
    --fake_dir /content/data/demo/fake/dalle_advanced \
    --out {WORK}/data/manifest_demo.csv --no_split --source official_demo

!cd /content/repo && python scripts/evaluate_robustness.py \
    --manifest {WORK}/data/manifest_demo.csv --split demo \
    --checkpoints clean={WORK}/checkpoints/detector_clean.pt \
                  robust={WORK}/checkpoints/detector_robust.pt \
    --out {WORK}/results_demo/ --limit 2000 --batch_size 64 --skip_compound

## Stage 8 — the demo, and the required `predict.py` output

In [ ]:
#@title 8a — predict.py on a folder (this is the required deliverable)
!cd /content/repo && python predict.py \
    --image_dir /content/data/demo/real/val2017 \
    --output {WORK}/results/predictions_demo.json \
    --checkpoint {WORK}/checkpoints/detector_robust.pt

import json
recs = json.load(open(WORK / "results/predictions_demo.json"))
print(json.dumps(recs[:5], indent=2))
print(f"\n{len(recs)} records, format = " + str(sorted(recs[0].keys())))

In [ ]:
#@title 8b — Gradio demo (record your video from this)
!cd /content/repo && python app/demo.py \
    --checkpoint {WORK}/checkpoints/detector_robust.pt --share

## Stage 9 — collect everything for the submission

In [ ]:
#@title 9 — copy results back into the repo and zip them
import shutil, pathlib
REPO_DIR = pathlib.Path('/content/repo')
for src, dst in [
    (WORK / "results/robustness.csv",        REPO_DIR / "results/robustness.csv"),
    (WORK / "results/summary.csv",           REPO_DIR / "results/summary.csv"),
    (WORK / "results/robustness_auc.png",    REPO_DIR / "results/robustness_auc.png"),
    (WORK / "results/degradation_auc.png",   REPO_DIR / "results/degradation_auc.png"),
    (WORK / "results/score_distributions.png", REPO_DIR / "results/score_distributions.png"),
    (WORK / "results/false_positives.png",   REPO_DIR / "results/false_positives.png"),
    (WORK / "results/false_negatives.png",   REPO_DIR / "results/false_negatives.png"),
    (WORK / "results/robustness_auc.md",     REPO_DIR / "results/robustness_auc.md"),
    (WORK / "results/headline.txt",          REPO_DIR / "results/headline.txt"),
    (WORK / "docs/ERROR_ANALYSIS.md",        REPO_DIR / "docs/ERROR_ANALYSIS.md"),
]:
    if pathlib.Path(src).exists():
        pathlib.Path(dst).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        print("copied", dst.name)

# The trained checkpoint is small enough to attach to a GitHub Release.
!ls -lh {WORK}/checkpoints/*.pt
!cd {WORK} && zip -qr submission_artifacts.zip results docs checkpoints/detector_robust.pt
print("\nNow: commit results/ and docs/ from /content/repo, and attach "
      "detector_robust.pt to a GitHub Release so predict.py works for reviewers.")